# Stage 05-08 + RACAF: Joint Training

**Status:** Repurposed as the joint Stage 05-08 + RACAF training notebook (locked --
`JOINT_TRAINING_ARCHITECTURE.md` §27). CORN (Stage 08) has no standalone training path of its
own -- none of Stage 05/06/07/RACAF/CORN does -- so this notebook, not four/five separate ones,
is the actual trainable-checkpoint boundary for all five. No second, competing joint-training
notebook exists or should be created.

## Objective

Train Stage 05 (Local Feature Extraction), Stage 06 (Global Feature Extraction), Stage 07
(Adaptive Cross-Attention), RACAF's trainable parameters, and CORN **jointly, as one model**,
through CORN's ordinal loss (`corn.corn_loss`) alone -- no auxiliary loss. Full design:
`JOINT_TRAINING_ARCHITECTURE.md`.

## Expected Inputs

- APTOS 2019's authoritative train/val split (`dataset_splits/aptos2019_train_val_split.csv`,
  2929 train / 733 val)
- Frozen Stage 1 (unused here), Stage 3 (vessel, `.pth`), Stage 4 (lesion, Experiment 2C `.keras`)
  checkpoints, already trained, resolved from Google Drive

## Expected Outputs (once real training is actually run -- not by this notebook as committed)

- A joint checkpoint (weights-only, per `JOINT_TRAINING_ARCHITECTURE.md` §25) under
  `experiments/FinalClassification/<timestamp>/checkpoints/` on Drive
- Best-checkpoint selection on maximum validation QWK

## Datasets

APTOS 2019 only. EyeQ and IDRiD are not read by this notebook.

## Dependencies

Stage 1-4 (frozen, loaded not trained), Stage 5/6/7/RACAF/CORN (all implemented, unit-tested,
composed here into one model -- see `joint_training_model.py`/`joint_training_dataset.py`).

## Current Implementation Status

Infrastructure, joint model/dataset code, and the actual `Trainer.fit()` training call are
all implemented and unit-tested (`tests/test_joint_training.py`). **`RUN_TRAINING = False`
throughout this notebook as committed -- opening or running every cell here does not start
real training or produce a checkpoint.** Setting `RUN_TRAINING = True` and re-running is a
separate, explicit, future action, never triggered by this commit.

---

Read `PROJECT_CODE.md`'s Development Workflow and `JOINT_TRAINING_ARCHITECTURE.md` in full before
changing anything below.


### Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s module docstring for why this is intentionally duplicated.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

### Imports

The reusable `colab/common/` infrastructure, the joint dataset loader (`joint_training_dataset.py`), and the joint model builder (`joint_training_model.py`) are all implemented -- see `JOINT_TRAINING_ARCHITECTURE.md`. `compile_joint_model()` compiles with `corn.corn_loss` as the sole training objective plus `corn.CORNQuadraticWeightedKappa` as a reported metric (never a second loss), so Keras produces `"QWK"`/`"val_QWK"` for this notebook's `monitor="val_QWK", mode="max"` checkpoint-selection policy below.


In [ ]:
import setup

setup_info = setup.setup()

import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)

import config
import corn
import joint_training_dataset as jtd
import joint_training_model as jtm
import racaf


### Dataset verification

APTOS2019 only -- EyeQ/IDRiD are not read by this notebook. Uses the existing, generic `verify_dataset.verify_image_folder()` (no dataset-specific verifier for APTOS2019 exists yet); `train.csv`'s presence and row count are checked directly.

In [ ]:
import os
import csv

import verify_dataset

aptos_raw_dir = config.dataset_raw_dir("APTOS2019")
train_csv_path = os.path.join(aptos_raw_dir, "train.csv")
train_image_dir = os.path.join(aptos_raw_dir, "train_images")

if not os.path.exists(train_csv_path):
    raise FileNotFoundError(
        f"APTOS2019 train.csv not found at {train_csv_path} -- verify Drive is mounted and "
        "APTOS2019_RAW_DIR resolved correctly (see colab_config.APTOS2019_RAW_DIR)."
    )
with open(train_csv_path, newline="", encoding="utf-8") as handle:
    labeled_row_count = sum(1 for _ in csv.DictReader(handle))
print(f"APTOS2019 train.csv: {labeled_row_count} labeled rows (expected 3662).")

verify_dataset.verify_image_folder(train_image_dir, min_images=labeled_row_count)
print("APTOS2019 train_images/ verified.")


### Frozen Stage 1/3/4 checkpoint discovery

Stage 1 is resolved for completeness only -- it is **not** loaded into the downstream APTOS graph (`JOINT_TRAINING_ARCHITECTURE.md` §3.1, locked). Stage 3/Stage 4 checkpoints must already exist on Drive; this notebook never trains, retrains, moves, renames, or overwrites them.

In [ ]:
print("Stage 1 (IQA) checkpoint  :", config.IQA_MODEL_DIR, "-- resolved only, NOT used in this graph")
print("Stage 3 (vessel) checkpoint:", config.VESSEL_SEG_MODEL_DIR)
print("Stage 4 (lesion) checkpoint:", config.LESION_SEG_MODEL_DIR)

vessel_checkpoint_path = os.path.join(config.VESSEL_SEG_MODEL_DIR, "best_model.pth")
lesion_checkpoint_path = os.path.join(config.LESION_SEG_MODEL_DIR, "best_model.keras")

for label, path in (("Stage 3 vessel", vessel_checkpoint_path), ("Stage 4 lesion (Experiment 2C)", lesion_checkpoint_path)):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} checkpoint not found at {path} -- see JOINT_TRAINING_ARCHITECTURE.md \u00a78.")
    print(f"{label} checkpoint found: {path}")


### Authoritative split

The SAME split Stage 5/6 already use -- `downstream_split.get_authoritative_split()`. No second split is computed here.

In [ ]:
train_entries, val_entries = jtd.split_train_val_ids()
print(f"train: {len(train_entries)} (expected 2929)")
print(f"val:   {len(val_entries)} (expected 733)")
assert len(train_entries) == 2929 and len(val_entries) == 733, "authoritative split counts do not match the committed manifest"


### Persistent cache locations

Stage 3/4's canonical (512×512) prediction cache and RACAF's `kappa`/`r` cache -- both persistent on Drive (`JOINT_TRAINING_ARCHITECTURE.md` §7.1/§10-12), populated once per image, never recomputed on a cache hit. This cell only reports the resolved paths -- it does not populate the cache itself. Populating it happens either explicitly via Phase 1 (`RUN_CACHE_PRECOMPUTATION` below -- recommended for a real training run) or lazily, on-the-fly per uncached image, when the dataset is actually iterated during training (gated behind `RUN_TRAINING` further below).

In [ ]:
print("Stage 3/4 canonical cache:", config.LOCAL_FEATURE_RESULTS_DIR)
print("RACAF reliability cache :", config.RACAF_RESULTS_DIR)


### Phase 1 -- Cache precomputation (gated -- separate from training)

Runs Stage 03/04 + RACAF inference for every entry in the authoritative split whose cache
isn't already populated, writing results to a **local** cache directory only (fast SSD I/O,
never per-image Drive round trips). An entry already cached from a prior run -- on Drive, at
`config.LOCAL_FEATURE_RESULTS_DIR`/`config.RACAF_RESULTS_DIR` -- is detected via a cheap
existence check (never a bulk content copy: see `JOINT_TRAINING_ARCHITECTURE.md` §35 for why
bulk-pulling the entire existing Drive cache down first previously crashed Drive's FUSE mount
with `OSError: [Errno 107] Transport endpoint is not connected`) and is skipped without being
recomputed or copied anywhere.

Set `CACHE_DIAGNOSTIC_MAX_IMAGES` to a small number (e.g. 5/10/25/50) to first exercise this
real code path -- real checkpoints, real images, real cache I/O -- on only that many images,
with per-image RSS/GPU-memory/timing logged, before committing to the full run.

Newly computed local entries are NOT automatically pushed to Drive -- run the "Phase 1b" cell
below (any time, safe to re-run) once you're ready to persist them.

In [ ]:
RUN_CACHE_PRECOMPUTATION = False  # set True to run Phase 1 (safe to interrupt and re-run)
CACHE_DIAGNOSTIC_MAX_IMAGES = None  # e.g. 5/10/25/50 -- run Phase 1 on only this many images first

LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
# Canonical RGB shares LOCAL_CACHE_DIR/persistent_cache_dir with Stage 03/04 -- no separate
# location. A local, real-code-path diagnostic showed regenerating it (raw read + Stage 02 + a
# skimage resize) from scratch, for all 3662 images, costs tens of minutes of CPU time on every
# fresh runtime when kept local-only, since /content is wiped between runtimes -- far more than
# the marginal Drive storage/read cost of persisting it like every other cache entry.
# Deliberately empty/nonexistent -- makes _resolve_processed_rgb() always take its cheap,
# unmodified live Stage 02 fallback instead of one more per-image Drive lookup.
NO_PRECOMPUTED_STAGE02_DIR = "/content/cache/_no_precomputed_stage02_output"

if RUN_CACHE_PRECOMPUTATION:
    import dataset_staging

    staged_aptos = dataset_staging.stage_dataset(colab_config.APTOS2019_RAW_DIR, "APTOS2019")
    dataset_staging.verify_staged_copy(staged_aptos)
    staged_train_image_dir = os.path.join(staged_aptos.local_dir, "train_images")

    # Cache hits are checked directly against the persistent Drive cache (existence-only, a
    # cheap stat per file) -- NEVER a bulk content copy. Newly computed entries are written to
    # local SSD only; push them to Drive with the "Phase 1b" cell below when ready.
    cache_stats = jtd.precompute_authoritative_joint_caches(
        image_dir=staged_train_image_dir,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
        max_images=CACHE_DIAGNOSTIC_MAX_IMAGES,
        verbose_diagnostics=CACHE_DIAGNOSTIC_MAX_IMAGES is not None,
    )
    print("Cache precomputation complete:", cache_stats)
else:
    print("RUN_CACHE_PRECOMPUTATION is False -- no dataset was staged, no Stage 03/04 inference was run, no cache was written.")

### Phase 1a -- Cache diagnostic (gated, measurement only)

Runs the **real, unmodified** `precompute_joint_frozen_caches()` on a handful of images and
reports, per image and per artifact (vessel / lesion / RACAF reliability / canonical RGB),
whether it was a local hit, a persistent (Drive) hit that got mirrored, or a genuine
recomputation -- plus the Drive stat/read counts, local write counts and elapsed time behind
each verdict.

Nothing is simulated: no synthetic cache, no mocked Drive, no fabricated timing, no simplified
stand-in for a cache function. `joint_cache_diagnostics.py` only wraps the real functions with
counters and timers that call straight through and return the real result, and it never writes
to, deletes from or modifies the persistent Drive cache -- a tripwire in the report asserts that
from the recorded operations rather than assuming it.

This does **not** run the full 3662-image Phase 1 and does **not** start training. It measures
only `DIAGNOSTIC_MAX_IMAGES` images (default 5), then repeats the same images a second time to
show whether pass 1 leaves a complete local working set behind.


In [ ]:
RUN_CACHE_DIAGNOSTIC = False  # set True to measure; leaves the full Phase 1 and training untouched
DIAGNOSTIC_MAX_IMAGES = 5     # how many images to measure -- 5 is plenty to answer the question
DIAGNOSTIC_IMAGE_IDS = None   # e.g. ["abc123", "def456"] to force specific ids (a known empty-FOV
                              # one from a real Phase 1 warning is a useful pick); overrides
                              # DIAGNOSTIC_MAX_IMAGES when set

if RUN_CACHE_DIAGNOSTIC:
    import dataset_staging
    import joint_cache_diagnostics as jcd

    # Same staging call Phase 1 uses -- idempotent, so this is a no-op if the dataset is already
    # staged in this runtime.
    staged_aptos = dataset_staging.stage_dataset(colab_config.APTOS2019_RAW_DIR, "APTOS2019")
    dataset_staging.verify_staged_copy(staged_aptos)
    staged_train_image_dir = os.path.join(staged_aptos.local_dir, "train_images")

    # Exactly the arguments the Phase 1 cell above passes -- the point is to measure THAT
    # configuration, so any divergence here would measure something else.
    diagnostic_report = jcd.run_diagnostic(
        entries=train_entries + val_entries,
        image_dir=staged_train_image_dir,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
        max_images=DIAGNOSTIC_MAX_IMAGES,
        explicit_ids=DIAGNOSTIC_IMAGE_IDS,
        # Reported read-only (path/existence/filesystem). Never created, never resolved through
        # experiment_manager -- the diagnostic does not touch experiment semantics.
        experiment_dir=colab_config.DRIVE.experiment_dir("FinalClassification"),
    )
    jcd.print_report(diagnostic_report)
else:
    print("RUN_CACHE_DIAGNOSTIC is False -- nothing was staged, measured, computed or written.")


### Phase 1b -- Manual flush to Drive (optional, safe to run any time)

Pushes whatever the local cache directories currently hold up to Drive, without running any inference. Useful right after manually interrupting a long Phase 1 run (`Runtime > Interrupt execution`) so that session's progress is not lost -- run this cell immediately afterward, before disconnecting. Also safe (and a no-op) to run after Phase 1 already finished and pushed everything itself.

In [ ]:
import dataset_staging

# Flushes everything Phase 1 wrote locally -- vessel/lesion/RACAF caches AND canonical RGB, since
# all four now share LOCAL_CACHE_DIR/config.LOCAL_FEATURE_RESULTS_DIR. Only genuinely new local
# entries are copied; anything already on Drive (e.g. every persistent-cache hit Phase 1 just
# mirrored) is left alone (JOINT_TRAINING_ARCHITECTURE.md §38).

flushed_lf, _kept_lf, failed_lf = dataset_staging.sync_missing_files(LOCAL_CACHE_DIR, config.LOCAL_FEATURE_RESULTS_DIR)
flushed_racaf, _kept_racaf, failed_racaf = dataset_staging.sync_missing_files(LOCAL_RACAF_CACHE_DIR, config.RACAF_RESULTS_DIR)
print(f"Flushed {flushed_lf} Stage 03/04 cache files and {flushed_racaf} RACAF cache files to Drive.")
if failed_lf or failed_racaf:
    print(f"{len(failed_lf) + len(failed_racaf)} file(s) failed to flush and remain local-only -- re-run this cell to retry them.")

### Joint model construction

Composes each stage's own, unmodified `build_*()` function -- `joint_training_model.build_joint_model()` -- into one functional model. No architecture is duplicated or modified here.

In [ ]:
joint_model = jtm.build_joint_model()
jtm.compile_joint_model(joint_model)

print("Total trainable parameters:", joint_model.count_params())
print("Loss:", joint_model.loss.__name__)
joint_model.summary(line_length=120)


### Smoke test (synthetic tensors only -- no real data, no training)

Confirms the joint graph builds, runs a forward pass, and backpropagates on THIS runtime/GPU before any real data is touched. Matches `tests/test_joint_training.py`'s own smoke test exactly.

In [ ]:
import numpy as np
import tensorflow as tf

_smoke_batch = 2
_s5 = np.random.rand(_smoke_batch, 512, 512, 8).astype("float32")
_s6 = np.random.rand(_smoke_batch, 256, 256, 3).astype("float32")
_r = np.random.rand(_smoke_batch, 1).astype("float32")
_grades = tf.constant([1, 3], dtype=tf.int32)

_logits = joint_model.predict([_s5, _s6, _r], verbose=0)
assert _logits.shape == (_smoke_batch, 4), _logits.shape

with tf.GradientTape() as _tape:
    _out = joint_model([_s5, _s6, _r], training=True)
    _loss = jtm.joint_corn_loss(_grades, _out)
_grads = _tape.gradient(_loss, joint_model.trainable_variables)
_none_grads = sum(1 for g in _grads if g is None)
assert _none_grads == 0, f"{_none_grads} trainable variables received no gradient"

print(f"Smoke test passed -- logits {_logits.shape}, loss {float(_loss):.4f}, "
      f"{len(joint_model.trainable_variables)} trainable variables, 0 missing gradients.")


### Training configuration

**`RUN_TRAINING = False`** -- running every cell above (and below) does NOT start real training
or produce a checkpoint. The dataset-loading and `Trainer.fit()` cells below are gated behind
this flag and print a message instead of running when it is `False`.

Initial T4 configuration (`JOINT_TRAINING_ARCHITECTURE.md` §24): `batch_size=2`,
`mixed_precision=True` -- a starting point, not a guaranteed-fitting value; verify empirically
before increasing. Best-checkpoint selection: `monitor="val_QWK"`, `mode="max"`.

In [ ]:
RUN_TRAINING = False  # DO NOT SET True IN THIS COMMIT -- see JOINT_TRAINING_ARCHITECTURE.md

BATCH_SIZE = 2
MIXED_PRECISION = True
EPOCHS = 50
MONITOR_METRIC = "val_QWK"
MONITOR_MODE = "max"
RESUME_EXPERIMENT_DIR = None  # set to an existing experiments/FinalClassification/<timestamp> to resume

print(f"RUN_TRAINING={RUN_TRAINING}  batch_size={BATCH_SIZE}  mixed_precision={MIXED_PRECISION}  "
      f"monitor={MONITOR_METRIC}/{MONITOR_MODE}")


### Dataset loading (gated -- requires real, Drive-mounted APTOS2019 data)

Stages APTOS locally (idempotent -- a no-op if Phase 1 already staged it this session) and
points the training pipeline at LOCAL cache/image directories, with the persistent Drive
cache as a read-only fallback for any entry not yet mirrored locally -- never per-epoch Drive
reads for an image/cache entry already available locally (`JOINT_TRAINING_ARCHITECTURE.md`
§36; a real run without this fix measured ~5-6s/step, traced to every sample reading its raw
image and all three Stage 03/04/RACAF cache files directly from Drive's FUSE mount, every
single epoch). Recommended: run Phase 1 first so this only pays Drive's per-file latency once
per image (mirroring to local disk) rather than during the first training epoch itself --
though this cell works correctly either way (any still-uncached entry is computed on the fly,
exactly as before).

In [ ]:
if RUN_TRAINING:
    import dataset_staging

    # Same local cache/staging locations as the Phase 1 cell above (idempotent to redefine/
    # re-stage here even if that cell already ran this session) -- so a cache Phase 1 already
    # built locally is reused, and this cell also works standalone if Phase 1 was skipped.
    LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
    LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
    NO_PRECOMPUTED_STAGE02_DIR = "/content/cache/_no_precomputed_stage02_output"

    staged_aptos = dataset_staging.stage_dataset(colab_config.APTOS2019_RAW_DIR, "APTOS2019")
    dataset_staging.verify_staged_copy(staged_aptos)
    staged_train_image_dir = os.path.join(staged_aptos.local_dir, "train_images")

    train_ds, val_ds = jtd.load_joint_training_datasets(
        batch_size=BATCH_SIZE,
        image_dir=staged_train_image_dir,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
    )
    print("Joint train/val tf.data pipelines built.")
else:
    print("RUN_TRAINING is False -- dataset pipelines were NOT built. Set RUN_TRAINING = True "
          "(and re-run) only when real joint training is actually intended.")

### Experiment + training (gated -- NOT executed merely by opening this notebook)

Uses the existing `experiment_manager.py`/`training.Trainer` infrastructure unmodified -- no new timestamp/run system, no new checkpoint format. When `RUN_TRAINING = True`, this cell calls `Trainer.fit(joint_model, train_ds, val_ds)` -- the project's existing training API -- which is the actual training execution. Best-checkpoint selection (`monitor="val_QWK", mode="max"`), resume (`RESUME_EXPERIMENT_DIR`), mixed precision, early stopping, and LR reduction are all handled by `Trainer`/`TrainingConfig`, unmodified.

In [ ]:
if RUN_TRAINING:
    import experiment_manager
    from training import Trainer, TrainingConfig

    experiment = experiment_manager.resolve_experiment(
        colab_config.DRIVE.experiment_dir("FinalClassification"), colab_config.REPO_DIR,
        resume_from=RESUME_EXPERIMENT_DIR,
        batch_size=BATCH_SIZE, epochs=EPOCHS, monitor=MONITOR_METRIC, mode=MONITOR_MODE,
    )

    training_config = TrainingConfig(
        run_dir=experiment.root, epochs=EPOCHS, monitor=MONITOR_METRIC, mode=MONITOR_MODE,
        mixed_precision=MIXED_PRECISION, resume=RESUME_EXPERIMENT_DIR is not None,
    )
    trainer = Trainer(training_config)

    print("Experiment resolved:", experiment.root)
    print(f"Training joint_model: {EPOCHS} epochs, batch_size={BATCH_SIZE}, "
          f"monitor={MONITOR_METRIC}/{MONITOR_MODE}, resume={training_config.resume}")

    # The ONLY real training call in this notebook. joint_model was already built and compiled
    # above (jtm.build_joint_model() + jtm.compile_joint_model() -- loss=joint_corn_loss only,
    # metrics=[corn.CORNQuadraticWeightedKappa()], no auxiliary loss, no class weighting);
    # train_ds/val_ds were built above from the authoritative APTOS2019 split (2929 train / 733
    # val, jtd.load_joint_training_datasets()). Trainer.fit() is the project's existing,
    # unmodified training API (training/trainer.py) -- it handles mixed precision, checkpointing
    # (best/last, monitor="val_QWK", mode="max", persisted under experiment.root/checkpoints/ on
    # Drive), early stopping, LR reduction, TensorBoard, and resume (via training_config.resume +
    # RESUME_EXPERIMENT_DIR) internally; none of that is reimplemented here. No class_weight is
    # passed (Trainer.fit()'s default, None) -- no class weighting is introduced.
    history = trainer.fit(joint_model, train_ds, val_ds)

    # tensorboard/ has no natural producer under Trainer's fixed logs/ convention -- archived
    # once training completes, per experiment_manager.py's own documented convention.
    experiment_manager.archive_tensorboard_logs(experiment)

    print("Training complete. Best checkpoint (max val_QWK):", trainer.best_weights_path())
    trainer.evaluate(joint_model, val_ds)  # EarlyStopping(restore_best_weights=True) already
                                            # restored the best epoch's weights into joint_model.
else:
    print("RUN_TRAINING is False -- no experiment directory was created, no Trainer was built, "
          "no dataset was iterated, no training was started.")
